# Using script to create subset

In [1]:
import sys
sys.path.append('../')
from create_subset_script import create_subset

In [2]:
df = create_subset([['sex','==','F']], 
                  save_path='preprocess/', 
                  save_df=False, 
                  return_df=True, 
                  preserve_meta_data=False,
                  features='Functional',
                  balance_classes=True, 
                  max_samples=10
                 )

[['sex', '==', 'F']]
Additional requirements:   balanced_ 10_samples
10 samples kept overall


In [3]:
path=create_subset([['sex','==','F']], 
                  save_path='./', 
                  save_df=True, 
                  return_df=False, 
                  preserve_meta_data=True,
                  features='Functional',
                  balance_classes=True, 
                  max_samples=2,
                  prefix='test_kk_'
                 )

[['sex', '==', 'F']]
Additional requirements:  meta_ balanced_ 2_samples
2 samples kept overall
Filtered dataframe saved to: ./test_kk_filtered_Functional_features_sex_==_F_meta_balanced_2_samples.json


# Explore Data

and metadata

In [1]:
import json
import pandas as pd
import numpy as np
import os
import stat

In [2]:
features = 'functional'

def preprocess_index(s):
    last_two_parts_of_the_path = '/'.join(s.split('/')[-2:])
    last_two_parts_of_the_path_without_file_extension = last_two_parts_of_the_path.split('.')[0]
    common_path = last_two_parts_of_the_path_without_file_extension.strip('_annot')
    return common_path

if features.lower() == 'functional':
    features_path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/zeidler/too_big_for_git/preprocess/ALC_features_Functional.json'
elif features.lower() == 'lld':
    features_path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/zeidler/too_big_for_git/preprocess/ALC_features_LLD.json'

features_df = pd.read_json(features_path, orient='index')
features_df['common_path'] = features_df.index.map(preprocess_index)

#absolute path so that we're not worried about where to run the script from
meta_data_path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/zeidler/intoxicat/data/meta_data_annotation_all_features_130623.json'

meta_data_df = pd.read_json(meta_data_path, orient='index')
meta_data_df['common_path'] = meta_data_df.index.map(preprocess_index)

In [3]:
meta_data_df.columns

Index(['path', 'name', 'annotates', 'sampleRate', 'utterance', 'utt', 'spn',
       'o_utt', 'item', 'o_item', 'alc', 'sex', 'age', 'acc', 'drh', 'aak',
       'bak', 'ges', 'ces', 'wea', 'irreg', 'anncom', 'specom', 'type',
       'content', 'wei', 'hei', 'edu', 'pro', 'smo', 'com', 'common_path'],
      dtype='object')

In [4]:
print('Example of common_path:', meta_data_df['common_path'][2])
meta_data_df.groupby(['common_path']).size().reset_index(name='counts')['counts'].unique()
#this means common_path is associated with one data sample each time and is safe to be used as index

Example of common_path: ses4038/5444038035_h_00


array([1])

In [5]:
print('Example of common_path:', features_df['common_path'][2])
features_df.groupby(['common_path']).size().reset_index(name='counts')['counts'].unique() #same as above

Example of common_path: ses4038/5444038035_h_00


array([1])

In [6]:
features_df = features_df.set_index('common_path')
meta_data_df = meta_data_df.set_index('common_path')
meta_data_df.head(1)

,path,name,annotates,sampleRate,utterance,utt,spn,o_utt,item,o_item,...,anncom,specom,type,content,wei,hei,edu,pro,smo,com
common_path,,,,,,,,,,,,,,,,,,,,,
ses4038/5444038020_h_00,/mount/arbeitsdaten/studenten1/team-lab-phonet...,5444038020_h_00,5444038020_h_00.wav,44100,5444038020_h_00,5444038020,544,5443046020,20,020,...,null,null,R,T,82,183,Abitur,jurist,N,null\n


In [5]:
#further exploration of columns in this section below
for column in meta_data_df.columns:
    print(f'meta_data name: {column}')
    print(f'value: {meta_data_df[column][0]}')

meta_data name: path
value: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/data/ALC/ses4038
meta_data name: name
value: 5444038020_h_00
meta_data name: annotates
value: 5444038020_h_00.wav
meta_data name: sampleRate
value: 44100
meta_data name: utterance
value: 5444038020_h_00
meta_data name: utt
value: 5444038020
meta_data name: spn
value: 544
meta_data name: o_utt
value: 5443046020
meta_data name: item
value: 20
meta_data name: o_item
value: 020
meta_data name: alc
value: na
meta_data name: sex
value: M
meta_data name: age
value: 27
meta_data name: acc
value: BY
meta_data name: drh
value: moderate
meta_data name: aak
value: 0.0
meta_data name: bak
value: 0.0
meta_data name: ges
value: f5
meta_data name: ces
value: r1
meta_data name: wea
value: SUN
meta_data name: irreg
value: 1|0|0|0|0|1|0|0|0
meta_data name: anncom
value: null
meta_data name: specom
value: null
meta_data name: type
value: R
meta_data name: content
value: T
meta_data name: wei
value: 82
meta_data name: hei
va

In [8]:
# columns_to_balance = ['spn', 'content'] #speaker id and content - perfectly balanced
# columns_to_balance = ['sex', 'content', 'acc', 'age', 'drh', 'smo',] #well balanced: important speaker characteristics
# #(we assume that edu and pro, as well as height, weight and weather don't really play a role )
# columns_to_balance = ['sex', 'type', 'acc', 'age', 'drh', 'smo',] #less perfect (type instead of content)
# columns_to_balance = ['sex', 'type', 'acc',] #even less perfect

In [9]:
for column in features_df.columns:
    print(f'features_data name: {column}')
    print(f'value: {features_df[column][0]}')

features_data name: intoxicated
value: na
features_data name: breath alcohol concentration
value: 0.0
features_data name: blood alcohol concentration
value: 0.0
features_data name: features
value: {'F0semitoneFrom27.5Hz_sma3nz_amean': [27.53673553466797], 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm': [0.23661102354526503], 'F0semitoneFrom27.5Hz_sma3nz_percentile20.0': [24.957035064697266], 'F0semitoneFrom27.5Hz_sma3nz_percentile50.0': [27.387535095214844], 'F0semitoneFrom27.5Hz_sma3nz_percentile80.0': [30.50052833557129], 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2': [5.543493270874023], 'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope': [281.2876281738281], 'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope': [335.7664794921875], 'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope': [170.46115112304688], 'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope': [131.8852081298828], 'loudness_sma3_amean': [0.37501123547554005], 'loudness_sma3_stddevNorm': [0.935931205749511], 'loudness_sma3_percentile20.0': [0.04

In [13]:
#This cell and the cell below are meant for selecting smaller subsets, not implemented yet

def randomly_select(values, n=1):
    '''
    returns a list of 'n' randomly selected distinct values from a given list of values
    at most the number of unique values in the given 'values' list is returned, even if 'n' is larger
    randomly_select(['544', '545', '545'], 10) => ['545', '544']
    '''
    import random
    random.seed(42)
    output = []
    out_n = min(len(set(values)), n)
    while len(output) < out_n:
        new_value = random.choice(values)
        if new_value not in output:
            output.append(new_value)
    return output

In [111]:
'''
Subset 0: one speaker, one task (content group)
'''
speakers = randomly_select(meta_data_df['spn'], 1)
print(f'Speakers: {speakers}') #530
condition = (meta_data_df['spn'].isin(speakers))
filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
print(f'Selected n samples: {len(filtered_df)}')

Speakers: [530]
Selected n samples: 90


In [15]:
filtered_df.shape

(90, 37)

# Counts per Speaker

Check the number of datapoints in each task group (or in each content type) for every speaker for every target.

Attention: takes a couple of minutes to calculate

In [10]:
%%time
all_speakers = meta_data_df['spn'].unique()
all_samples_per_speaker = dict()
content_groups_per_speaker = dict()
types_per_speaker = dict()

for speaker in all_speakers:
    condition = (meta_data_df['spn'] == speaker)
    filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
    all_samples_per_speaker[str(speaker)] = len(filtered_df)
    
    per_content_group_subdict = {}
    for content_group in meta_data_df['content'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['content'] == content_group)
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        per_content_group_subdict[content_group] = len(filtered_df)
    content_groups_per_speaker[str(speaker)] = per_content_group_subdict
    
    per_type_subdict = {}
    for content_type in meta_data_df['type'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['type'] == content_type)
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        per_type_subdict[content_type] = len(filtered_df)
    types_per_speaker[str(speaker)] = per_type_subdict

CPU times: user 29.8 s, sys: 114 ms, total: 29.9 s
Wall time: 30.2 s


In [17]:
all_targets_per_speaker = dict()
for speaker in all_speakers:
    a_condition = (meta_data_df['spn'] == speaker) & (meta_data_df['alc'] == 'a')
    a_filtered_df = meta_data_df[a_condition].join(features_df, how='left', on='common_path', lsuffix='_l')

    na_condition = (meta_data_df['spn'] == speaker) & (meta_data_df['alc'] == 'na')
    na_filtered_df = meta_data_df[na_condition].join(features_df, how='left', on='common_path', lsuffix='_l')

    cna_condition = (meta_data_df['spn'] == speaker) & (meta_data_df['alc'] == 'cna')
    cna_filtered_df = meta_data_df[cna_condition].join(features_df, how='left', on='common_path', lsuffix='_l')
    all_targets_per_speaker[str(speaker)] = {'a': len(a_filtered_df), 
                                             'na': len(na_filtered_df), 
                                             'cna': len(cna_filtered_df)}

In [18]:
a_content_groups_per_speaker = dict()
a_types_per_speaker = dict()
for speaker in all_speakers:
    a_content_groups_per_speaker[str(speaker)] = {}
    for content_group in meta_data_df['content'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['content'] == content_group) & (meta_data_df['alc'] == 'a')
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        a_content_groups_per_speaker[str(speaker)][content_group] = len(filtered_df)
    
    a_types_per_speaker[str(speaker)] = {}
    for content_type in meta_data_df['type'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['type'] == content_type) & (meta_data_df['alc'] == 'a')
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        a_types_per_speaker[str(speaker)][content_type] = len(filtered_df)
    

In [20]:
na_content_groups_per_speaker = dict()
na_types_per_speaker = dict()
for speaker in all_speakers:
    na_content_groups_per_speaker[str(speaker)] = {}
    for content_group in meta_data_df['content'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['content'] == content_group) & (meta_data_df['alc'] == 'na')
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        na_content_groups_per_speaker[str(speaker)][content_group] = len(filtered_df)
    
    na_types_per_speaker[str(speaker)] = {}
    for content_type in meta_data_df['type'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['type'] == content_type) & (meta_data_df['alc'] == 'na')
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        na_types_per_speaker[str(speaker)][content_type] = len(filtered_df)
    

In [21]:
cna_content_groups_per_speaker = dict()
cna_types_per_speaker = dict()
for speaker in all_speakers:
    cna_content_groups_per_speaker[str(speaker)] = {}
    for content_group in meta_data_df['content'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['content'] == content_group) & (meta_data_df['alc'] == 'cna')
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        cna_content_groups_per_speaker[str(speaker)][content_group] = len(filtered_df)
    
    cna_types_per_speaker[str(speaker)] = {}
    for content_type in meta_data_df['type'].unique():
        condition = (meta_data_df['spn'] == speaker) & (meta_data_df['type'] == content_type) & (meta_data_df['alc'] == 'cna')
        filtered_df = meta_data_df[condition].join(features_df, how='left', on='common_path', lsuffix='_l')
        cna_types_per_speaker[str(speaker)][content_type] = len(filtered_df)
        

In [22]:
a_stats = {'content_groups': a_content_groups_per_speaker,
           'types': a_types_per_speaker}
na_stats = {'content_groups': na_content_groups_per_speaker,
           'types': na_types_per_speaker}
cna_stats = {'content_groups': cna_content_groups_per_speaker,
           'types': cna_types_per_speaker}
stats = {'a': a_stats, 'na': na_stats, 'cna': cna_stats}

In [23]:
import json
with open('stats/samples_per_speaker.json', 'w') as jsn:
    json.dump(all_samples_per_speaker, jsn)
with open('stats/content_groups_per_speaker.json', 'w') as jsn:
    json.dump(content_groups_per_speaker, jsn)
with open('stats/types_per_speaker.json', 'w') as jsn:
    json.dump(types_per_speaker, jsn)
with open('stats/all_targets_per_speaker.json', 'w') as jsn:
    json.dump(all_targets_per_speaker, jsn)
with open('stats/all_targets_content_groups_types.json', 'w') as jsn:
    json.dump(stats, jsn)

In [24]:
def s(x):
    #sort a dict by values
    return {k: v for k, v in sorted(x.items(), key=lambda item: item[1])}

In [25]:
from collections import defaultdict
all_samples_per_speaker_counts = defaultdict(int)
for k, v in all_samples_per_speaker.items():
    all_samples_per_speaker_counts[v] += 1
    
for k, v in all_samples_per_speaker_counts.items():
    print(f'{v} speakers recorded {k} datapoints each')

142 speakers recorded 90 datapoints each
20 speakers recorded 120 datapoints each


In [26]:
content_groups_per_speaker_counts = defaultdict(int)
for k, v in content_groups_per_speaker.items():
    v = str(list(v.values()))
    content_groups_per_speaker_counts[v] += 1

content_groups = list(list(content_groups_per_speaker.values())[0].keys())

for k, v in content_groups_per_speaker_counts.items():
    print(f'{v} speakers recorded this number of recordings per content group: {k}')
print(f'with the following content groups: {content_groups}')

142 speakers recorded this number of recordings per content group: [15, 15, 15, 2, 15, 13, 6, 9]
20 speakers recorded this number of recordings per content group: [20, 20, 20, 3, 20, 17, 8, 12]
with the following content groups: ['T', 'A', 'N', 'S', 'C', 'R', 'Q', 'P']


In [27]:
types_per_speaker_counts = defaultdict(int)
for k, v in types_per_speaker.items():
    v = str(list(v.values()))
    types_per_speaker_counts[v] += 1

types = list(list(types_per_speaker.values())[0].keys())

for k, v in types_per_speaker_counts.items():
    print(f'{v} speakers recorded this number of recordings per type: {k}')
print(f'with the following content groups: {types}')

142 speakers recorded this number of recordings per type: [41, 19, 15, 7, 8]
20 speakers recorded this number of recordings per type: [54, 26, 20, 9, 11]
with the following content groups: ['R', 'L', 'E', 'D', 'M']


In [28]:
all_targets_per_speaker_counts = defaultdict(int)
for k, v in all_targets_per_speaker.items():
    v = str(list(v.values()))
    all_targets_per_speaker_counts[v] += 1

types = ('a', 'na', 'cna')

for k, v in all_targets_per_speaker_counts.items():
    print(f'{v} speakers recorded this number of recordings per target: {k}')
print(f'with the following targets: {types}')

142 speakers recorded this number of recordings per target: [30, 60, 0]
20 speakers recorded this number of recordings per target: [30, 60, 30]
with the following targets: ('a', 'na', 'cna')


In [29]:
for t in ('a', 'na', 'cna'):
    print(t)
    t_types_per_speaker = stats[t]['types'] #e.g. alc=a, 
    t_types_per_speaker_counts = defaultdict(int)
    for k, v in t_types_per_speaker.items():
        v = str(list(v.values()))
        t_types_per_speaker_counts[v] += 1

    types = list(list(t_types_per_speaker.values())[0].keys())

    for k, v in t_types_per_speaker_counts.items():
        print(f'{v} speakers recorded this number of recordings per type: {k}')
    print(f'with the following types: {types}')

a
162 speakers recorded this number of recordings per type: [13, 7, 5, 2, 3]
with the following types: ['R', 'L', 'E', 'D', 'M']
na
162 speakers recorded this number of recordings per type: [28, 12, 10, 5, 5]
with the following types: ['R', 'L', 'E', 'D', 'M']
cna
142 speakers recorded this number of recordings per type: [0, 0, 0, 0, 0]
20 speakers recorded this number of recordings per type: [13, 7, 5, 2, 3]
with the following types: ['R', 'L', 'E', 'D', 'M']


In [30]:
for t in ('a', 'na', 'cna'):
    print(t)
    t_content_groups_per_speaker = stats[t]['content_groups']
    t_content_groups_per_speaker_counts = defaultdict(int)
    for k, v in t_content_groups_per_speaker.items():
        v = str(list(v.values()))
        t_content_groups_per_speaker_counts[v] += 1

    content_groups = list(list(t_content_groups_per_speaker.values())[0].keys())

    for k, v in t_content_groups_per_speaker_counts.items():
        print(f'{v} speakers recorded this number of recordings per content group: {k}')
    print(f'with the following content groups: {content_groups}')

a
162 speakers recorded this number of recordings per content group: [5, 5, 5, 1, 5, 4, 2, 3]
with the following content groups: ['T', 'A', 'N', 'S', 'C', 'R', 'Q', 'P']
na
162 speakers recorded this number of recordings per content group: [10, 10, 10, 1, 10, 9, 4, 6]
with the following content groups: ['T', 'A', 'N', 'S', 'C', 'R', 'Q', 'P']
cna
142 speakers recorded this number of recordings per content group: [0, 0, 0, 0, 0, 0, 0, 0]
20 speakers recorded this number of recordings per content group: [5, 5, 5, 1, 5, 4, 2, 3]
with the following content groups: ['T', 'A', 'N', 'S', 'C', 'R', 'Q', 'P']


In [31]:
#safe check
a_check = [5, 5, 5, 1, 5, 4, 2, 3]
na_check = [10, 10, 10, 1, 10, 9, 4, 6]
cna_check = [5, 5, 5, 1, 5, 4, 2, 3]
ns = (162, 162, 20)

total = 0
for idx, _ in enumerate((a_check, na_check, cna_check)):
    for __ in _:
        total += __ * ns[idx]
total

15180

In [32]:
len(features_df)

15180

In [33]:
#safe check
a_check = [13, 7, 5, 2, 3]
na_check = [28, 12, 10, 5, 5]
cna_check = [13, 7, 5, 2, 3]
ns = (162, 162, 20)

total = 0
for idx, _ in enumerate((a_check, na_check, cna_check)):
    for __ in _:
        total += __ * ns[idx]
total

15180

In [34]:
len(features_df)

15180

# Balance

In [9]:
'''Subset 1: largest target-and-speaker-balanced subset without duplicates and without cna
Contains 162 speakers, where, for each target in 'a', 'na':
content group distribution ['T', 'A', 'N', 'S', 'C', 'R', 'Q', 'P']: [5, 5, 5, 1, 5, 4, 2, 3]
this will result in unbalanced type distribution

Attention: there's more than one way to make this subset since we can disregard different 50% of 'na'
'''
content_group_names = ['T', 'A', 'N', 'S', 'C', 'R', 'Q', 'P']
content_group_counts_to_keep = [5, 5, 5, 1, 5, 4, 2, 3] #all 'a' counts and half of 'na' counts, as obtained above
m = {content_group_names[i]: content_group_counts_to_keep[i] for i in range(len(content_group_names))}

#select subset of intoxicated (all intoxicated samples); we start with 'a' since it is smaller
condition_a = (meta_data_df['alc'] == 'a')
filtered_df_a = meta_data_df[condition_a].join(features_df, how='left', lsuffix='_l') #on index common_path, set above

balanced_filtered_df = filtered_df_a
#now for each content group, select the same amount of samples for each speaker as for intoxicated
#append after selecting
all_speakers = meta_data_df['spn'].unique()
for speaker in all_speakers:
    for content_group in meta_data_df['content'].unique():
        condition_spn_group = (meta_data_df['spn'] == speaker) & (meta_data_df['content'] == content_group) & (meta_data_df['alc'] == 'na')
        filtered_df_spn_group = meta_data_df[condition_spn_group].join(features_df, how='left', lsuffix='_l')
        #keep the same amount as in 'a' target
        filtered_df_spn_group = filtered_df_spn_group.sample(n=m[content_group], random_state=42) #randomly keeps n rows for speaker and content group
        #append
        balanced_filtered_df = pd.concat([balanced_filtered_df, filtered_df_spn_group])

#shuffle the rows        
filtered_df = balanced_filtered_df.sample(frac=1, random_state=42) 
print(f'{len(filtered_df)} samples kept overall')        

9720 samples kept overall


In [10]:
filtered_df.head()

,path,name,annotates,sampleRate,utterance,utt,spn,o_utt,item,o_item,...,wei,hei,edu,pro,smo,com,intoxicated,breath alcohol concentration,blood alcohol concentration,features
common_path,,,,,,,,,,,,,,,,,,,,,
ses3042/5413042019_h_00,/mount/arbeitsdaten/studenten1/team-lab-phonet...,5413042019_h_00,5413042019_h_00.wav,44100,5413042019_h_00,5413042019,541,5414047019,19,019,...,65,170,Abitur,jurist,Y,null\n,a,0.00130,0.00132,{'F0semitoneFrom27.5Hz_sma3nz_amean': [36.7825...
ses2042/0392042001_h_00,/mount/arbeitsdaten/studenten1/team-lab-phonet...,0392042001_h_00,0392042001_h_00.wav,44100,0392042001_h_00,392042001,39,0391040001,1,001,...,55,160,Abitur,judical clerk,N,null\n,na,0.00000,0.00000,{'F0semitoneFrom27.5Hz_sma3nz_amean': [32.7760...
ses1051/0501051017_h_00,/mount/arbeitsdaten/studenten1/team-lab-phonet...,0501051017_h_00,0501051017_h_00.wav,44100,0501051017_h_00,501051017,50,0502040024,17,024,...,57,166,Abitur,assessor,N,null\n,a,0.00058,0.00062,{'F0semitoneFrom27.5Hz_sma3nz_amean': [38.1266...
ses1062/0611062018_h_00,/mount/arbeitsdaten/studenten1/team-lab-phonet...,0611062018_h_00,0611062018_h_00.wav,44100,0611062018_h_00,611062018,61,0612052038,18,038,...,70,156,Abitur,clerk,N,null\n,a,0.00080,0.00125,{'F0semitoneFrom27.5Hz_sma3nz_amean': [32.1210...
ses2045/0552045027_h_00,/mount/arbeitsdaten/studenten1/team-lab-phonet...,0552045027_h_00,0552045027_h_00.wav,44100,0552045027_h_00,552045027,55,null,27,null,...,56,165,Abitur,assessor,N,null\n,na,0.00000,0.00000,{'F0semitoneFrom27.5Hz_sma3nz_amean': [34.9757...


In [15]:
#check: each speaker has equal amount of data samples in each target
print(filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts')['counts'].unique())
filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts')

[30]


,spn,alc,counts
0,6,a,30
1,6,na,30
2,7,a,30
3,7,na,30
4,8,a,30
...,...,...,...
319,594,na,30
320,595,a,30
321,595,na,30
322,596,a,30


In [22]:
#check: each speaker has equal amount of data samples in each target
print(filtered_df.groupby(['spn', 'content']).size().reset_index(name='counts')['counts'].unique())
for content_group in ['A', 'R', 'S', 'T', 'N', 'P', 'Q', 'C']:
    print(content_group)
    print('different counts')
    print(filtered_df[filtered_df['content']==content_group].groupby(['spn', 'content']).size().reset_index(name='counts')['counts'].unique())
    print('different counts with alc=a')
    print(filtered_df[(filtered_df['content']==content_group) & (filtered_df['alc']=='a')].groupby(['spn', 'content']).size().reset_index(name='counts')['counts'].unique())

[10  6  4  8  2]
A
different counts
[10]
different counts with alc=a
[5]
R
different counts
[8]
different counts with alc=a
[4]
S
different counts
[2]
different counts with alc=a
[1]
T
different counts
[10]
different counts with alc=a
[5]
N
different counts
[10]
different counts with alc=a
[5]
P
different counts
[6]
different counts with alc=a
[3]
Q
different counts
[4]
different counts with alc=a
[2]
C
different counts
[10]
different counts with alc=a
[5]


In [33]:
#check: each speaker has equal amount of data samples in each target
print(filtered_df.groupby(['spn', 'type']).size().reset_index(name='counts')['counts'].unique())
for content_type in ['E', 'M', 'D', 'R', 'L']:
    print(content_type)
    print('different counts')
    print(filtered_df[filtered_df['type']==content_type].groupby(['spn', 'type']).size().reset_index(name='counts')['counts'].unique())
    print('different counts with alc=a')
    print(filtered_df[(filtered_df['type']==content_type) & (filtered_df['alc']=='a')].groupby(['spn', 'type']).size().reset_index(name='counts')['counts'].unique())

[ 4 10 14  6 26]
E
different counts
[10]
different counts with alc=a
[5]
M
different counts
[6]
different counts with alc=a
[3]
D
different counts
[4]
different counts with alc=a
[2]
R
different counts
[26]
different counts with alc=a
[13]
L
different counts
[14]
different counts with alc=a
[7]


In [26]:
filtered_df.groupby(['type', 'content']).size().reset_index(name='counts')

,type,content,counts
0,D,P,322
1,D,Q,402
2,E,C,1620
3,L,N,1620
4,L,S,324
5,L,T,250
6,M,P,650
7,M,Q,246
8,R,A,1620
9,R,R,1296


In [28]:
print(meta_data_df.groupby(['type', 'content']).size().reset_index(name='counts'))

   type content  counts
0     D       P     506
1     D       Q     668
2     E       C    2530
3     L       N    2530
4     L       S     344
5     L       T     344
6     M       P    1012
7     M       Q     344
8     R       A    2530
9     R       R    2186
10    R       T    2186


In [17]:
#sanity check: same amount of samples in each type for each speaker in the 'a' and 'na' parts of the subset
#UPD: not the same, because types and content groups are not balanced
from collections import defaultdict
a_types_per_speaker = dict()
for speaker in all_speakers:   
    a_types_per_speaker[str(speaker)] = {}
    for content_type in meta_data_df['type'].unique():
        condition = (filtered_df['spn'] == speaker) & (filtered_df['type'] == content_type) & (filtered_df['alc'] == 'a')
        filtered_df_a = filtered_df[condition]
        a_types_per_speaker[str(speaker)][content_type] = len(filtered_df_a)


na_types_per_speaker = dict()
for speaker in all_speakers:   
    na_types_per_speaker[str(speaker)] = {}
    for content_type in meta_data_df['type'].unique():
        condition = (filtered_df['spn'] == speaker) & (filtered_df['type'] == content_type) & (filtered_df['alc'] == 'na')
        filtered_df_na = filtered_df[condition]
        na_types_per_speaker[str(speaker)][content_type] = len(filtered_df_na)

a_types_per_speaker_counts = defaultdict(int)
for k, v in a_types_per_speaker.items():
    v = str(list(v.values()))
    a_types_per_speaker_counts[v] += 1

a_types = list(list(a_types_per_speaker.values())[0].keys())

for k, v in a_types_per_speaker_counts.items():
    print(f'target: a. {v} speakers recorded this number of recordings per type: {k}')
    print(f'with the following content types: {a_types}')

na_types_per_speaker_counts = defaultdict(int)
for k, v in na_types_per_speaker.items():
    v = str(list(v.values()))
    na_types_per_speaker_counts[v] += 1

na_types = list(list(na_types_per_speaker.values())[0].keys())

for k, v in na_types_per_speaker_counts.items():
    print(f'target: na. {v} speakers recorded this number of recordings per type: {k}')
    print(f'with the following content types: {na_types}')
    

target: a. 162 speakers recorded this number of recordings per type: [13, 7, 5, 2, 3]
with the following content types: ['R', 'L', 'E', 'D', 'M']
target: na. 39 speakers recorded this number of recordings per type: [13, 7, 5, 2, 3]
with the following content types: ['R', 'L', 'E', 'D', 'M']
target: na. 10 speakers recorded this number of recordings per type: [14, 6, 5, 4, 1]
with the following content types: ['R', 'L', 'E', 'D', 'M']
target: na. 35 speakers recorded this number of recordings per type: [13, 7, 5, 3, 2]
with the following content types: ['R', 'L', 'E', 'D', 'M']
target: na. 6 speakers recorded this number of recordings per type: [13, 7, 5, 4, 1]
with the following content types: ['R', 'L', 'E', 'D', 'M']
target: na. 25 speakers recorded this number of recordings per type: [14, 6, 5, 2, 3]
with the following content types: ['R', 'L', 'E', 'D', 'M']
target: na. 28 speakers recorded this number of recordings per type: [14, 6, 5, 3, 2]
with the following content types: ['R',

In [92]:
#condition = (filtered_df['spn'] == 544) & (filtered_df['type'] == 'R') & (filtered_df['alc'] == 'na')
condition = filtered_df['alc'] == 'a'
(filtered_df[condition]['type']).describe()

count     4860
unique       5
top          R
freq      2106
Name: type, dtype: object

In [93]:
condition = meta_data_df['alc'] == 'a'
(meta_data_df[condition]['type']).describe()

count     4860
unique       5
top          R
freq      2106
Name: type, dtype: object

In [24]:
def save_df(filtered_df, out_filename):
    import json
    with open(out_filename, 'w') as jsn:
        json.dump(filtered_df.to_dict(orient='index'), jsn)
    print(f'Filtered dataframe saved to: {out_filename}')

In [25]:
save_df(filtered_df, 'subsets/subset_1.json')

Filtered dataframe saved to: subsets/subset_1.json


In [29]:
'''Subset 2: largest target-and-speaker-balanced subset without duplicates and without cna
Contains 162 speakers, where, for each target in 'a', 'na':
type distribution ['R', 'L', 'E', 'D', 'M']: [13, 7, 5, 2, 3]
this will result in unbalanced content group distribution

Attention: there's more than one way to make this subset since we can disregard different 50% of 'na'
'''
type_names = ['R', 'L', 'E', 'D', 'M']
type_counts_to_keep = [13, 7, 5, 2, 3] #all 'a' counts and half of 'na' counts, as obtained above
m = {type_names[i]: type_counts_to_keep[i] for i in range(len(type_names))}

#select subset of intoxicated (all intoxicated samples); we start with 'a' since it is smaller
condition_a = (meta_data_df['alc'] == 'a')
filtered_df_a = meta_data_df[condition_a].join(features_df, how='left', lsuffix='_l') #on index common_path, set above

balanced_filtered_df = filtered_df_a
#now for each content group, select the same amount of samples for each speaker as for intoxicated
#append after selecting
all_speakers = meta_data_df['spn'].unique()
for speaker in all_speakers:
    for content_type in meta_data_df['type'].unique():
        condition_spn_group = (meta_data_df['spn'] == speaker) & (meta_data_df['type'] == content_type) & (meta_data_df['alc'] == 'na')
        filtered_df_spn_group = meta_data_df[condition_spn_group].join(features_df, how='left', lsuffix='_l')
        #keep the same amount as in 'a' target
        filtered_df_spn_group = filtered_df_spn_group.sample(n=m[content_type], random_state=42) #randomly keeps n rows for speaker and content group
        #append
        balanced_filtered_df = pd.concat([balanced_filtered_df, filtered_df_spn_group])

#shuffle the rows        
filtered_df = balanced_filtered_df.sample(frac=1, random_state=42) 
print(f'{len(filtered_df)} samples kept overall')        

9720 samples kept overall


In [30]:
#check: each speaker has equal amount of data samples in each target
print(filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts')['counts'].unique())
filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts')

[30]


,spn,alc,counts
0,6,a,30
1,6,na,30
2,7,a,30
3,7,na,30
4,8,a,30
...,...,...,...
319,594,na,30
320,595,a,30
321,595,na,30
322,596,a,30


In [36]:
#content groups for each type (possible counts per speaker)
print(filtered_df.groupby(['spn', 'content']).size().reset_index(name='counts')['counts'].unique())
for content_group in ['A', 'R', 'S', 'T', 'N', 'P', 'Q', 'C']:
    print(content_group)
    print('different counts')
    print(filtered_df[filtered_df['content']==content_group].groupby(['spn', 'content']).size().reset_index(name='counts')['counts'].unique())
    print('different counts with alc=a')
    print(filtered_df[(filtered_df['content']==content_group) & (filtered_df['alc']=='a')].groupby(['spn', 'content']).size().reset_index(name='counts')['counts'].unique())

[11 10  7  3  2  9 12  8  1  5  6  4 13 14]
A
different counts
[11  9 10  7 12  8  6 13]
different counts with alc=a
[5]
R
different counts
[10  8  6  9  5  7 11 12]
different counts with alc=a
[4]
S
different counts
[2 1]
different counts with alc=a
[1]
T
different counts
[ 7 10 12  9 11 13  8 14]
different counts with alc=a
[5]
N
different counts
[10 12 11]
different counts with alc=a
[5]
P
different counts
[7 5 6 8]
different counts with alc=a
[3]
Q
different counts
[3 5 4 2]
different counts with alc=a
[2]
C
different counts
[10]
different counts with alc=a
[5]


In [35]:
#check: each speaker has equal amount of data samples in each target
print(filtered_df.groupby(['spn', 'type']).size().reset_index(name='counts')['counts'].unique())
for content_type in ['E', 'M', 'D', 'R', 'L']:
    print(content_type)
    print('different counts')
    print(filtered_df[filtered_df['type']==content_type].groupby(['spn', 'type']).size().reset_index(name='counts')['counts'].unique())
    print('different counts with alc=a')
    print(filtered_df[(filtered_df['type']==content_type) & (filtered_df['alc']=='a')].groupby(['spn', 'type']).size().reset_index(name='counts')['counts'].unique())

[ 4 10 14  6 26]
E
different counts
[10]
different counts with alc=a
[5]
M
different counts
[6]
different counts with alc=a
[3]
D
different counts
[4]
different counts with alc=a
[2]
R
different counts
[26]
different counts with alc=a
[13]
L
different counts
[14]
different counts with alc=a
[7]


In [37]:
save_df(filtered_df, 'subsets/subset_2.json')

Filtered dataframe saved to: subsets/subset_2.json


In [ ]:
'''
Easier part: task specific (i.e. with one content type) datasets, balanced

R|L -> subset_read_list.json
E|M|D -> subset_free.json
'''